# Frequency Model — Claim Count Distribution

This notebook models the **claim frequency** per policyholder in a French MTPL portfolio.

The goal is to select and calibrate a discrete distribution that best fits the observed claim counts.

## 1. Imports and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson, nbinom
from sklearn.metrics import r2_score

df = pd.read_csv("freMTPL2freq.csv")
claim_counts = df["ClaimNb"]
n_policyholders = claim_counts.count()

print(f"Number of policyholders : {n_policyholders}")
print(f"Min claims              : {claim_counts.min()}")
print(f"Q25                     : {np.percentile(claim_counts, 25)}")
print(f"Q75                     : {np.percentile(claim_counts, 75)}")
print(f"Max claims              : {claim_counts.max()}")

## 2. Exploratory Analysis

In [ ]:
plt.hist(claim_counts, bins=16)
plt.title("Claim Count Distribution")
plt.xlabel("Number of Claims")
plt.ylabel("Frequency")
plt.show()

In [ ]:
freq_mean = claim_counts.mean()
freq_var = claim_counts.var()

print(f"Mean     : {freq_mean:.6f}")
print(f"Variance : {freq_var:.6f}")
print(f"\nVariance > Mean : {freq_var > freq_mean} — overdispersion detected")

The variance exceeds the mean, which indicates **overdispersion**. This suggests the Poisson distribution may not be the best fit — we will compare it against the Negative Binomial.

## 3. Observed Frequencies

In [ ]:
total_claims = np.sum(claim_counts)
lambda_poisson = total_claims / n_policyholders

observed_counts = claim_counts.value_counts().reindex(
    range(claim_counts.max() + 1), fill_value=0
)
observed_mass = observed_counts / n_policyholders

print(f"Total claims    : {total_claims}")
print(f"Lambda (Poisson): {lambda_poisson:.6f}")

## 4. Poisson Fit

As a baseline, we fit a Poisson distribution with parameter λ estimated by the empirical mean.

In [ ]:
poisson_mass = pd.Series(
    [poisson.pmf(k, lambda_poisson) for k in range(claim_counts.max() + 1)]
)
poisson_counts = poisson_mass * n_policyholders

r2_poisson = r2_score(observed_counts, poisson_counts)

# Error at k=0
error_at_zero_poisson = (observed_counts[0] - poisson_counts[0]) / observed_counts[0]

print(f"R² Poisson      : {r2_poisson:.6f}")
print(f"%Error at k=0    : {error_at_zero_poisson *100:.4f}%")

plt.plot(observed_mass, color='green', label='Observed')
plt.plot(poisson_mass, color='red', alpha=0.5, label='Poisson')
plt.title("Observed vs Poisson")
plt.xlabel("Number of Claims")
plt.ylabel("Probability Mass")
plt.legend()
plt.show()

## 5. Negative Binomial Fit

Given the overdispersion, we fit a **Negative Binomial** distribution using moment matching:

$$p = \frac{\mu}{\sigma^2}, \quad n = \frac{\mu \cdot p}{1 - p}$$

In [ ]:
p_nbinom = freq_mean / freq_var
n_nbinom = freq_mean * p_nbinom / (1 - p_nbinom)

print(f"Negative Binomial parameters:")
print(f"  n : {n_nbinom:.4f}")
print(f"  p : {p_nbinom:.4f}")

nbinom_mass = pd.Series(
    [nbinom.pmf(k, n_nbinom, p_nbinom) for k in range(int(claim_counts.max()) + 1)]
)
nbinom_counts = nbinom_mass * n_policyholders

r2_nbinom = r2_score(observed_mass, nbinom_mass)

# Error at k=0
error_at_zero_nbinom = (observed_counts[0] - nbinom_counts[0]) / observed_counts[0]

print(f"\nR² Negative Binomial : {r2_nbinom:.8f}")
print(f"%Error at k=0    : {error_at_zero_nbinom *100:.4f}%")

## 6. Model Comparison

In [ ]:
comparison = pd.DataFrame({
    "Observed"         : observed_counts,
    "Negative Binomial": nbinom_counts.round(),
    "Diff NB"          : observed_counts - nbinom_counts.round(),
    "Poisson"          : poisson_counts.round(),
    "Diff Poisson"     : observed_counts - poisson_counts.round()
})
print(comparison)

plt.plot(observed_counts, color='green', label='Observed')
plt.plot(nbinom_counts, color='blue', label='Negative Binomial')
plt.plot(poisson_counts, color='red', alpha=0.5, label='Poisson')
plt.title("Observed vs Fitted Distributions")
plt.xlabel("Number of Claims")
plt.ylabel("Count")
plt.legend()
plt.show()

## 7. Conclusion

The **Negative Binomial** distribution outperforms the Poisson on all metrics:
- Higher R²
- Better fit at k=0 (policyholders with no claims)
- Accounts for overdispersion (variance > mean)

**Selected model** : Negative Binomial with parameters estimated by moment matching.

These parameters (`n_nbinom`, `p_nbinom`) are carried forward into `aggregate_loss.ipynb`.